# Retail Revelations — Project 2

Customer retention + cohort analysis using `online_retail_II.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
DATA_PATH = Path('online_retail_II.csv')  # update if needed
OUT = Path('data_processed'); OUT.mkdir(exist_ok=True)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH, encoding_errors='ignore')
df.columns = [str(c).strip().replace(' ', '').replace('_','').lower() for c in df.columns]
rename = {}
if 'invoiceno' in df.columns: rename['invoiceno'] = 'invoice'
if 'unitprice' in df.columns: rename['unitprice'] = 'price'
if 'customerid' not in df.columns and 'customerid' in df.columns: rename['customerid'] = 'customerid'
df = df.rename(columns=rename)
df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors='coerce')


In [ ]:
clean = df.dropna(subset=['customerid','invoicedate','invoice']).copy()
clean['invoice'] = clean['invoice'].astype(str)
clean = clean[~clean['invoice'].str.startswith('C', na=False)].copy()
clean['quantity'] = pd.to_numeric(clean['quantity'], errors='coerce')
clean['price'] = pd.to_numeric(clean['price'], errors='coerce')
clean = clean.dropna(subset=['quantity','price'])
clean = clean[(clean['quantity']>0) & (clean['price']>0)].copy()
clean['customerid'] = pd.to_numeric(clean['customerid'], errors='coerce').astype('Int64')
clean = clean.dropna(subset=['customerid']).copy()
clean['customerid'] = clean['customerid'].astype(int)
clean['line_revenue'] = clean['quantity'] * clean['price']


In [ ]:
agg = {
  'invoicedate': ('invoicedate','min'),
  'customerid': ('customerid','first'),
  'items': ('stockcode','count'),
  'qty': ('quantity','sum'),
  'revenue': ('line_revenue','sum')
}
if 'country' in clean.columns:
    agg['country'] = ('country','first')
invoice_kpi = clean.groupby('invoice', as_index=False).agg(**agg)
invoice_kpi['month'] = invoice_kpi['invoicedate'].dt.to_period('M').dt.to_timestamp()
monthly = (invoice_kpi.groupby('month', as_index=False)
           .agg(orders=('invoice','nunique'), customers=('customerid','nunique'), revenue=('revenue','sum')))
monthly['aov'] = monthly['revenue'] / monthly['orders']
monthly['mom_growth'] = monthly['revenue'].pct_change()
monthly.to_csv(OUT/'monthly_kpis.csv', index=False)
invoice_kpi.to_csv(OUT/'invoice_kpis.csv', index=False)


In [ ]:
cust_value = (invoice_kpi.groupby('customerid', as_index=False)
              .agg(orders=('invoice','nunique'), revenue=('revenue','sum')))
cust_value['is_repeat_buyer'] = cust_value['orders'] >= 2
repeat_rate = cust_value['is_repeat_buyer'].mean()
cust_value.to_csv(OUT/'customer_value.csv', index=False)
repeat_rate


In [ ]:
invoice_kpi['order_month'] = invoice_kpi['invoicedate'].dt.to_period('M')
cohort = invoice_kpi.groupby('customerid')['order_month'].min().rename('cohort_month')
invoice_kpi = invoice_kpi.join(cohort, on='customerid')
invoice_kpi['month_index'] = (invoice_kpi['order_month'] - invoice_kpi['cohort_month']).apply(lambda p: p.n)
cohort_sizes = cohort.value_counts().rename_axis('cohort_month').reset_index(name='cohort_size')
activity = (invoice_kpi.groupby(['cohort_month','order_month','month_index'])['customerid']
            .nunique().rename('active_customers').reset_index())
retention = activity.merge(cohort_sizes, on='cohort_month', how='left')
retention['retention_rate'] = retention['active_customers'] / retention['cohort_size']
retention = retention.sort_values(['cohort_month','month_index'])
retention.to_csv(OUT/'cohort_retention_long.csv', index=False)
matrix = (retention[retention['month_index'].between(0,12)]
          .pivot(index='cohort_month', columns='month_index', values='retention_rate')
          .sort_index())
matrix.to_csv(OUT/'cohort_retention_matrix.csv')


In [ ]:
plt.figure()
plt.plot(monthly['month'], monthly['revenue'])
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG/'monthly_revenue.png', dpi=200)
plt.close()


In [ ]:
plt.figure()
vals = matrix.fillna(0).values
plt.imshow(vals, aspect='auto')
plt.title('Cohort Retention Heatmap (Rate)')
plt.xlabel('Month Index (0 = cohort month)')
plt.ylabel('Cohort Month')
plt.colorbar(label='Retention Rate')
plt.yticks(ticks=np.arange(len(matrix.index)), labels=[str(p) for p in matrix.index])
plt.xticks(ticks=np.arange(len(matrix.columns)), labels=[str(c) for c in matrix.columns])
plt.tight_layout()
plt.savefig(FIG/'cohort_retention_heatmap.png', dpi=200)
plt.close()
